# 02 · Fine-tuning RoBERTa-base & FinBERT

Fine-tunes two transformer models on Financial PhraseBank using HuggingFace Trainer
with fixed hyperparameters (lr 2e-5, batch 16, up to 5 epochs, early stopping on
macro F1). Also runs the learning curve sweep at training sizes
[100, 250, 500, 1000, full].

**Requires:** Google Colab GPU runtime (Runtime → Change runtime type → T4 GPU).

**Writes:** `results/main_comparison.json` (appends), `results/learning_curve.json`,
model checkpoints to `models/`.

In [ ]:
!pip install -q 'datasets>=2.14,<4.0' transformers accelerate torch scikit-learn numpy

## 1 · Configuration

In [ ]:
# ── Toggle this when running on Colab ──────────────────────────
USE_DRIVE  = False
DRIVE_PATH = '/content/drive/MyDrive/cs443-final-project'
# ───────────────────────────────────────────────────────────────

import json, random, time
import numpy as np
import torch
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path(DRIVE_PATH)
else:
    BASE = Path('.')

RESULTS_DIR = BASE / 'results'
MODELS_DIR  = BASE / 'models'
for d in [RESULTS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Fine-tuning will be very slow on CPU.')

## 2 · Label Scheme & Metrics Helpers

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

ID2LABEL    = {0: 'negative', 1: 'neutral', 2: 'positive'}
LABEL2ID    = {v: k for k, v in ID2LABEL.items()}
LABEL_NAMES = ['negative', 'neutral', 'positive']


def compute_metrics(y_true, y_pred):
    y_true, y_pred = list(y_true), list(y_pred)
    acc      = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average='macro',
                               labels=[0, 1, 2], zero_division=0))
    per_f1   = f1_score(y_true, y_pred, average=None,
                        labels=[0, 1, 2], zero_division=0)
    cm       = confusion_matrix(y_true, y_pred, labels=[0, 1, 2]).tolist()
    return {
        'accuracy':         acc,
        'macro_f1':         macro_f1,
        'per_class_f1':     {ID2LABEL[i]: float(per_f1[i]) for i in range(3)},
        'confusion_matrix': cm,
    }


def print_metrics(m, title=''):
    if title:
        print('\n' + '─' * 54)
        print('  ' + title)
        print('─' * 54)
    acc = m['accuracy']
    f1  = m['macro_f1']
    print(f'  Accuracy : {acc:.4f}   Macro F1: {f1:.4f}')
    for name, val in m['per_class_f1'].items():
        print(f'  {name:<10}: F1={val:.4f}')


def trainer_compute_metrics(eval_pred):
    """Adapter for HuggingFace Trainer's compute_metrics API."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    m = compute_metrics(labels.tolist(), preds.tolist())
    return {'accuracy': m['accuracy'], 'macro_f1': m['macro_f1']}

## 3 · Load & Split Financial PhraseBank

In [ ]:
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split

print('Loading Financial PhraseBank...')
raw = load_dataset('gtfintechlab/financial_phrasebank_sentences_allagree')
ds  = raw['train']
if 'sentence' in ds.column_names:
    ds = ds.rename_column('sentence', 'text')

texts  = list(ds['text'])
labels = list(ds['label'])

train_texts, tmp_texts, train_labels, tmp_labels = train_test_split(
    texts, labels, test_size=0.30, random_state=SEED, stratify=labels
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    tmp_texts, tmp_labels, test_size=0.50, random_state=SEED, stratify=tmp_labels
)

train_ds = Dataset.from_dict({'text': train_texts, 'label': train_labels})
val_ds   = Dataset.from_dict({'text': val_texts,   'label': val_labels})
test_ds  = Dataset.from_dict({'text': test_texts,  'label': test_labels})

print(f'  Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')

## 4 · Fine-tuning Helper

Hyperparameters are fixed and not tuned. Listed here as a limitation:

| Hyperparameter | Value |
|---|---|
| Learning rate | 2e-5 |
| Batch size | 16 |
| Max epochs | 5 |
| Early stopping patience | 2 epochs |
| Best checkpoint metric | macro F1 |
| Max sequence length | 128 |

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

MAX_LEN    = 128
LR         = 2e-5
BATCH_SIZE = 16
MAX_EPOCHS = 5


def tokenize_dataset(ds, tokenizer):
    tok = ds.map(
        lambda b: tokenizer(b['text'], padding='max_length',
                            truncation=True, max_length=MAX_LEN),
        batched=True, remove_columns=['text'],
    )
    tok.set_format('torch')
    return tok


def finetune(model_name, tr_ds, vl_ds, te_ds, output_dir):
    """Fine-tune model_name and return metrics dict + inference seconds."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f'  Loading tokeniser and model: {model_name}')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )

    print('  Tokenising splits...')
    tr_tok = tokenize_dataset(tr_ds, tokenizer)
    vl_tok = tokenize_dataset(vl_ds, tokenizer)
    te_tok = tokenize_dataset(te_ds, tokenizer)

    # Compute warmup_steps from warmup_ratio (avoids deprecation warning in transformers 5+)
    steps_per_epoch = max(1, len(tr_tok) // BATCH_SIZE)
    warmup_steps    = int(0.1 * steps_per_epoch * MAX_EPOCHS)

    args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=MAX_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        learning_rate=LR,
        warmup_steps=warmup_steps,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        greater_is_better=True,
        save_total_limit=1,
        logging_steps=50,
        seed=SEED,
        fp16=(device == 'cuda'),
        report_to='none',
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tr_tok,
        eval_dataset=vl_tok,
        compute_metrics=trainer_compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    print('  Training...')
    trainer.train()
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    print(f'  Checkpoint saved → {output_dir}')

    print('  Evaluating on test set...')
    t0   = time.perf_counter()
    pred = trainer.predict(te_tok)
    elapsed = time.perf_counter() - t0

    y_pred = np.argmax(pred.predictions, axis=-1).tolist()
    y_true = te_tok['label'].tolist()
    m = compute_metrics(y_true, y_pred)
    m['inference_seconds'] = round(elapsed, 4)
    m['num_examples']      = len(y_true)
    m['cost_per_1k']       = 0.0
    return m

## 5 · Fine-tune RoBERTa-base

In [ ]:
print('=' * 60)
print('Fine-tuning RoBERTa-base')
print('=' * 60)

roberta_dir = MODELS_DIR / 'roberta-base_phrasebank'
roberta_metrics = finetune(
    'roberta-base', train_ds, val_ds, test_ds, roberta_dir
)
print_metrics(roberta_metrics, 'RoBERTa-base — PhraseBank Test')

## 6 · Fine-tune FinBERT

In [ ]:
print('=' * 60)
print('Fine-tuning FinBERT (ProsusAI/finbert)')
print('=' * 60)

finbert_dir = MODELS_DIR / 'finbert_phrasebank'
finbert_metrics = finetune(
    'ProsusAI/finbert', train_ds, val_ds, test_ds, finbert_dir
)
print_metrics(finbert_metrics, 'FinBERT — PhraseBank Test')

## 7 · Save Full-Training Results

In [ ]:
results_path = RESULTS_DIR / 'main_comparison.json'
if results_path.exists():
    with open(results_path) as f:
        all_results = json.load(f)
else:
    all_results = {}

all_results['roberta-base'] = {
    'model': 'roberta-base',
    'model_name': 'roberta-base',
    'train_size': len(train_ds),
    'phrasebank_test': roberta_metrics,
}
all_results['finbert'] = {
    'model': 'finbert',
    'model_name': 'ProsusAI/finbert',
    'train_size': len(train_ds),
    'phrasebank_test': finbert_metrics,
}

with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print('Saved:', results_path)

## 8 · Learning Curve Sweep

Fine-tune both models at training sizes [100, 250, 500, 1000, full] and record
macro F1 on the PhraseBank test set. Subsampling is stratified so class balance
is preserved at every size.

In [ ]:
TRAIN_SIZES   = [100, 250, 500, 1000]   # 'full' is added from main results
LC_MODELS     = [
    ('roberta-base',       'roberta-base'),
    ('finbert',            'ProsusAI/finbert'),
]

lc_results = {short: [] for short, _ in LC_MODELS}

for short_name, model_name in LC_MODELS:
    print('\n' + '#' * 60)
    print(f'  Learning curve: {model_name}')
    print('#' * 60)

    train_labels_arr = np.array(train_labels)

    for size in TRAIN_SIZES:
        print(f'\n--- Training size: {size} ---')
        rng = np.random.default_rng(SEED)
        indices = []
        for cls in range(3):
            cls_idx = np.where(train_labels_arr == cls)[0]
            n = max(1, round(size * len(cls_idx) / len(train_labels_arr)))
            chosen = rng.choice(cls_idx, size=min(n, len(cls_idx)), replace=False)
            indices.extend(chosen.tolist())
        rng.shuffle(indices)
        indices = indices[:size]

        sub_texts  = [train_texts[i]  for i in indices]
        sub_labels = [train_labels[i] for i in indices]
        sub_ds = Dataset.from_dict({'text': sub_texts, 'label': sub_labels})

        out_dir = MODELS_DIR / f'{short_name}_phrasebank_{size}'
        m = finetune(model_name, sub_ds, val_ds, test_ds, out_dir)
        lc_results[short_name].append({
            'train_size': size,
            'macro_f1':   m['macro_f1'],
            'accuracy':   m['accuracy'],
        })
        print(f'  size={size}  macro_f1={m["macro_f1"]:.4f}')

    # Add full-training result from main_comparison.json
    mc_path = RESULTS_DIR / 'main_comparison.json'
    if mc_path.exists():
        with open(mc_path) as f:
            mc = json.load(f)
        if short_name in mc:
            full_m = mc[short_name]['phrasebank_test']
            lc_results[short_name].append({
                'train_size': len(train_ds),
                'macro_f1':   full_m['macro_f1'],
                'accuracy':   full_m['accuracy'],
            })

lc_path = RESULTS_DIR / 'learning_curve.json'
with open(lc_path, 'w') as f:
    json.dump(lc_results, f, indent=2)
print('\nLearning curve saved:', lc_path)